In [1]:
# Importamos las librerías necesarias para conectarnos a BigQuery.
import os
from pathlib import Path

from dotenv import load_dotenv
from google.cloud import bigquery

In [2]:
# Localizamos la raíz del proyecto.
# Este notebook está dentro de:
# parte_2_modelo_bigquery/notebooks/
PROJECT_ROOT = Path.cwd().parents[1]

# Cargamos las variables del archivo .env.
load_dotenv(PROJECT_ROOT / ".env", override=True)

# Obtenemos la configuración del proyecto.
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")

# Construimos la ruta absoluta de las credenciales.
CREDENTIALS_PATH = PROJECT_ROOT / os.getenv(
    "GOOGLE_APPLICATION_CREDENTIALS"
)

# Configuramos las credenciales para Google Cloud.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(
    CREDENTIALS_PATH
)

print("Proyecto:", PROJECT_ID)
print("Dataset:", DATASET_ID)
print("Credenciales:", CREDENTIALS_PATH)
print("¿Existe el archivo?:", CREDENTIALS_PATH.exists())

Proyecto: tc-sql-miguel
Dataset: electromarket
Credenciales: c:\Users\mgpir\Desktop\RepoPracticaObligatoria\bootcamp_AI_Engineering_05_26\tc-sql-lopezmiguel\credentials\service-account.json
¿Existe el archivo?: True


In [3]:
# Creamos el cliente de BigQuery.
client = bigquery.Client(project=PROJECT_ID)

print("Conexión con BigQuery correcta.")

Conexión con BigQuery correcta.


## Query 1 — Ingresos por mes

Calculamos los ingresos generados por las ventas agrupados por mes.
El importe de cada línea se obtiene utilizando la cantidad, el precio
histórico de compra y el descuento aplicado.

In [4]:
# Construimos la referencia completa al dataset.
DATASET_REF = f"{PROJECT_ID}.{DATASET_ID}"

print("Dataset utilizado:", DATASET_REF)

Dataset utilizado: tc-sql-miguel.electromarket


In [ ]:
# Calculamos los ingresos mensuales.
query_ingresos_mes = f"""
SELECT
    FORMAT_DATE('%Y-%m', o.order_date) AS mes,
    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount)
        ),
        2
    ) AS ingresos
FROM `{DATASET_REF}.orders` AS o
JOIN `{DATASET_REF}.order_items` AS oi
    ON o.order_id = oi.order_id
WHERE o.status != 'cancelled'
GROUP BY mes
ORDER BY mes;
"""

resultado_ingresos_mes = client.query(
    query_ingresos_mes
).to_dataframe()

resultado_ingresos_mes

## Query 2 — Productos más vendidos

Identificamos los productos con mayor número de unidades vendidas.
Se excluyen los pedidos cancelados.

In [5]:
# Obtenemos los productos más vendidos por unidades.
query_productos_vendidos = f"""
SELECT
    p.product_id,
    p.name AS producto,
    SUM(oi.quantity) AS unidades_vendidas
FROM `{DATASET_REF}.order_items` AS oi
JOIN `{DATASET_REF}.products` AS p
    ON oi.product_id = p.product_id
JOIN `{DATASET_REF}.orders` AS o
    ON oi.order_id = o.order_id
WHERE o.status != 'cancelled'
GROUP BY
    p.product_id,
    p.name
ORDER BY unidades_vendidas DESC
LIMIT 10;
"""

resultado_productos_vendidos = client.query(
    query_productos_vendidos
).to_dataframe()

resultado_productos_vendidos

C:\Users\mgpir\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_id,producto,unidades_vendidas
0,40,Power Bank jTq-964,175
1,60,ZenBook IuG-201,156
2,30,Xperia wOC-021,154
3,46,Gaming Ixt-624,153
4,15,Galaxy amD-608,152
5,58,Galaxy Watch OAZ-991,140
6,56,ThinkPad rHx-044,138
7,48,Inspiron HQQ-158,138
8,41,MacBook Jkz-328,137
9,64,Galaxy Watch IHk-084,137


## Query 3 — Clientes por país

Analizamos la distribución de clientes según su país de residencia.

In [6]:
# Contamos los clientes registrados por país.
query_clientes_pais = f"""
SELECT
    country AS pais,
    COUNT(*) AS numero_clientes
FROM `{DATASET_REF}.customers`
GROUP BY country
ORDER BY numero_clientes DESC;
"""

resultado_clientes_pais = client.query(
    query_clientes_pais
).to_dataframe()

resultado_clientes_pais

C:\Users\mgpir\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,pais,numero_clientes
0,Belgium,75
1,Portugal,69
2,Spain,68
3,Netherlands,60
4,France,60
5,Ireland,58
6,Italy,57
7,Germany,53


## Query 4 — Tiempo medio de entrega

Calculamos el tiempo medio transcurrido entre la fecha del pedido
y la fecha de entrega para los pedidos entregados.

In [7]:
# Calculamos el tiempo medio de entrega en días.
query_tiempo_entrega = f"""
SELECT
    ROUND(
        AVG(
            DATE_DIFF(
                delivered_date,
                order_date,
                DAY
            )
        ),
        2
    ) AS tiempo_medio_entrega_dias
FROM `{DATASET_REF}.orders`
WHERE status = 'delivered'
    AND delivered_date IS NOT NULL;
"""

resultado_tiempo_entrega = client.query(
    query_tiempo_entrega
).to_dataframe()

resultado_tiempo_entrega

C:\Users\mgpir\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,tiempo_medio_entrega_dias
0,191.43


## Query 5 — Ingresos y margen por categoría

Calculamos los ingresos y el margen bruto generado por cada categoría.
El margen se obtiene utilizando el precio de venta histórico y el coste
del producto.

In [8]:
# Calculamos ingresos y margen bruto por categoría.
query_margen_categoria = f"""
SELECT
    c.name AS categoria,

    ROUND(
        SUM(
            oi.quantity
            * oi.unit_price
            * (1 - oi.discount)
        ),
        2
    ) AS ingresos,

    ROUND(
        SUM(
            oi.quantity
            * (
                oi.unit_price * (1 - oi.discount)
                - p.cost
            )
        ),
        2
    ) AS margen_bruto

FROM `{DATASET_REF}.order_items` AS oi

JOIN `{DATASET_REF}.products` AS p
    ON oi.product_id = p.product_id

JOIN `{DATASET_REF}.categories` AS c
    ON p.category_id = c.category_id

JOIN `{DATASET_REF}.orders` AS o
    ON oi.order_id = o.order_id

WHERE o.status != 'cancelled'

GROUP BY c.name

ORDER BY margen_bruto DESC;
"""

resultado_margen_categoria = client.query(
    query_margen_categoria
).to_dataframe()

resultado_margen_categoria

C:\Users\mgpir\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,categoria,ingresos,margen_bruto
0,Wearables,1092265.16,207784.36
1,Laptops,1042075.26,202042.30
2,Gaming,803719.72,167601.09
3,Smartphones,761740.13,157319.92
4,Accessories,604163.88,144007.29
5,Peripherals,696848.13,131658.96
6,Audio,506081.28,85154.04


## Query 6 — Distribución de estados de pedidos

Analizamos cuántos pedidos existen actualmente en cada estado.
Esto permite detectar pedidos pendientes, cancelados, devueltos o entregados.

In [9]:
# Analizamos la distribución de pedidos según su estado.
query_estado_pedidos = f"""
SELECT
    status AS estado,
    COUNT(*) AS numero_pedidos
FROM `{DATASET_REF}.orders`
GROUP BY status
ORDER BY numero_pedidos DESC;
"""

resultado_estado_pedidos = client.query(
    query_estado_pedidos
).to_dataframe()

resultado_estado_pedidos

C:\Users\mgpir\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,estado,numero_pedidos
0,delivered,347
1,pending,342
2,returned,339
3,confirmed,325
4,cancelled,324
5,shipped,323


In [10]:
# Comprobamos que las siete tablas contienen registros.
tablas = [
    "categories",
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "reviews",
]

for tabla_nombre in tablas:
    tabla_ref = f"{DATASET_REF}.{tabla_nombre}"
    tabla = client.get_table(tabla_ref)

    print(
        f"{tabla_nombre}: {tabla.num_rows} registros"
    )

categories: 7 registros
customers: 500 registros
products: 70 registros
orders: 2000 registros
order_items: 5044 registros
payments: 2000 registros
reviews: 306 registros
